# SciFact Reranker Benchmark on Colab

This notebook benchmarks a fixed `embeddinggemma_fact_check` bi-encoder retriever plus a diverse set of rerankers on `mteb/scifact`.

It evaluates `query -> abstract` retrieval over the SciFact test set: `300` unique queries over `5,183` abstracts.


## Experiment setup

This experiment fixes the first-stage retriever to `embeddinggemma_fact_check`, then reranks the bi-encoder candidates at `K=50` and `K=100`.

Reranker shortlist:

- `cross-encoder/ms-marco-MiniLM-L12-v2`
- `Alibaba-NLP/gte-reranker-modernbert-base`
- `jinaai/jina-reranker-v2-base-multilingual`
- `BAAI/bge-reranker-v2-m3`
- `Qwen/Qwen3-Reranker-0.6B`

The report compares retriever-only against reranked results using `nDCG@10`, `MRR@10`, `MAP@10`, stage-1 candidate recall, reranker latency, and total query latency.


In [ ]:
# Optional: clone the repo only if you want a fresh remote checkout.
# Do not run this if you already uploaded or mounted an updated local copy.
# !git clone https://github.com/Wasiq-Malik/ScholarRAG.git /content/ScholarRAG


In [ ]:
%cd /content/ScholarRAG

!python -m pip install -U pip
!pip install -e .


In [1]:
!nvidia-smi || true


Wed Apr 29 01:02:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os

RUN_NAME = f"colab_scifact_rerank_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path("benchmarks/scifact/results") / RUN_NAME

# Optional: paste a Hugging Face token for gated models in this session only.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("HF_TOKEN is not set. embeddinggemma-based retrieval will fail until you provide it.")

RETRIEVER_MODEL = "embeddinggemma_fact_check"
RERANKERS = [
    "msmarco_minilm_l12_v2",
    "gte_reranker_modernbert_base",
    "jina_reranker_v2_base_multilingual",
    "bge_reranker_v2_m3",
    "qwen3_reranker_0_6b",
]
CANDIDATE_KS = [50, 100]
SPLIT = "test"
DOCUMENT_MODE = "abstract"
DEVICE = "cuda"
DTYPE = "auto"
MAX_QUERIES = None

GPU_PRESETS = {
    "l4_default": {
        "msmarco_minilm_l12_v2": 64,
        "gte_reranker_modernbert_base": 32,
        "jina_reranker_v2_base_multilingual": 16,
        "bge_reranker_v2_m3": 16,
        "qwen3_reranker_0_6b": 8,
    },
    "l4_conservative": {
        "msmarco_minilm_l12_v2": 32,
        "gte_reranker_modernbert_base": 16,
        "jina_reranker_v2_base_multilingual": 8,
        "bge_reranker_v2_m3": 8,
        "qwen3_reranker_0_6b": 4,
    },
}

GPU_PRESET = "l4_default"
RERANKER_BATCH_OVERRIDES = dict(GPU_PRESETS[GPU_PRESET])

print(json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "retriever_model": RETRIEVER_MODEL,
    "rerankers": RERANKERS,
    "candidate_ks": CANDIDATE_KS,
    "split": SPLIT,
    "document_mode": DOCUMENT_MODE,
    "device": DEVICE,
    "dtype": DTYPE,
    "gpu_preset": GPU_PRESET,
    "reranker_batch_overrides": RERANKER_BATCH_OVERRIDES,
    "hf_token_set": bool(os.environ.get("HF_TOKEN")),
}, indent=2))


{
  "output_dir": "benchmarks/scifact/results/colab_scifact_rerank_20260429_010501",
  "retriever_model": "embeddinggemma_fact_check",
  "rerankers": [
    "msmarco_minilm_l12_v2",
    "gte_reranker_modernbert_base",
    "jina_reranker_v2_base_multilingual",
    "bge_reranker_v2_m3",
    "qwen3_reranker_0_6b"
  ],
  "candidate_ks": [
    50,
    100
  ],
  "split": "test",
  "document_mode": "abstract",
  "device": "cuda",
  "dtype": "auto",
  "gpu_preset": "l4_default",
  "reranker_batch_overrides": {
    "msmarco_minilm_l12_v2": 64,
    "gte_reranker_modernbert_base": 32,
    "jina_reranker_v2_base_multilingual": 16,
    "bge_reranker_v2_m3": 16,
    "qwen3_reranker_0_6b": 8
  },
  "hf_token_set": true
}


In [3]:
from benchmarks.scifact.models import available_model_keys
from benchmarks.scifact.reranker_models import available_reranker_keys

retriever_models = available_model_keys()
reranker_models = available_reranker_keys()
print("Retriever models:", retriever_models)
print("Reranker models:", reranker_models)
assert RETRIEVER_MODEL in retriever_models
missing = sorted(set(RERANKERS) - set(reranker_models))
if missing:
    raise ValueError(f"Notebook reranker list is not supported by the checked-out code: {missing}")


ModuleNotFoundError: No module named 'benchmarks'

## Preset guidance

- Start with `l4_default`.
- `qwen3_reranker_0_6b` and `jina_reranker_v2_base_multilingual` are the first rerankers to reduce if memory is tight.
- Candidate `K=100` is the main setting to test because your current retriever already reaches `Recall@100 = 0.9833`.
- `K=50` is the latency/quality tradeoff check.


In [ ]:
import json
import os
import shlex
import signal
import subprocess
import sys
from pathlib import Path


def run_one_reranker(reranker_key: str) -> dict:
    model_dir = OUTPUT_DIR / reranker_key
    model_dir.mkdir(parents=True, exist_ok=True)
    log_path = model_dir / "run.log"

    cmd = [
        sys.executable,
        "benchmarks/scifact/run_reranker_benchmark.py",
        "--retriever-model", RETRIEVER_MODEL,
        "--split", SPLIT,
        "--document-mode", DOCUMENT_MODE,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--output-dir", str(model_dir),
        "--rerankers", reranker_key,
        "--candidate-ks", *[str(k) for k in CANDIDATE_KS],
    ]

    if MAX_QUERIES is not None:
        cmd += ["--max-queries", str(MAX_QUERIES)]

    batch_size = RERANKER_BATCH_OVERRIDES.get(reranker_key)
    if batch_size is not None:
        cmd += ["--reranker-batch-size", str(batch_size)]

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    print(f"
=== Running {reranker_key} ===")
    print("Command:")
    print(" ".join(shlex.quote(part) for part in cmd))
    print(f"Log file: {log_path}")
    print(f"Configured batch size: {batch_size}")

    with log_path.open("w") as log_handle:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
        returncode = process.wait()

    result = {
        "reranker": reranker_key,
        "batch_size": batch_size,
        "returncode": returncode,
        "status": "completed" if returncode == 0 else "failed",
        "log_path": str(log_path),
        "output_dir": str(model_dir),
    }
    if returncode == -signal.SIGKILL:
        result["failure_reason"] = "likely_oom_or_os_kill"
    elif returncode != 0:
        result["failure_reason"] = f"nonzero_exit_{returncode}"

    status_path = model_dir / "notebook_status.json"
    status_path.write_text(json.dumps(result, indent=2))
    return result

all_results = []
for reranker_key in RERANKERS:
    all_results.append(run_one_reranker(reranker_key))

aggregate_status_path = OUTPUT_DIR / "notebook_status.json"
aggregate_status_path.parent.mkdir(parents=True, exist_ok=True)
aggregate_status_path.write_text(json.dumps(all_results, indent=2))
all_results


In [ ]:
import json
import pandas as pd
from benchmarks.scifact.generate_reranker_report import build_report

summary_rows = []
for result in all_results:
    model_dir = Path(result["output_dir"])
    summary_path = model_dir / "summary.csv"
    if summary_path.exists():
        frame = pd.read_csv(summary_path)
        if not frame.empty:
            summary_rows.append(frame)

if summary_rows:
    aggregate_summary = pd.concat(summary_rows, ignore_index=True)
    aggregate_summary = aggregate_summary.sort_values(["ndcg_at_10", "avg_total_query_latency_ms"], ascending=[False, True])
    aggregate_summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
    (OUTPUT_DIR / "summary.json").write_text(aggregate_summary.to_json(orient="records", indent=2))

    aggregate_config = {
        "retriever_model": RETRIEVER_MODEL,
        "rerankers": RERANKERS,
        "candidate_ks": CANDIDATE_KS,
        "split": SPLIT,
        "document_mode": DOCUMENT_MODE,
    }
    (OUTPUT_DIR / "config.json").write_text(json.dumps(aggregate_config, indent=2))
    report_path = build_report(OUTPUT_DIR)
    print(f"Wrote aggregate report to {report_path}")
    aggregate_summary
else:
    print("No successful reranker runs produced a summary.csv file.")


In [ ]:
from IPython.display import Markdown, display

report_path = OUTPUT_DIR / "REPORT.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print("No aggregate report found yet.")


In [ ]:
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
for name in [
    "ndcg_at_10.png",
    "mrr_at_10.png",
    "map_at_10.png",
    "avg_total_query_latency_ms.png",
    "ndcg_vs_total_query_latency.png",
]:
    path = plots_dir / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))


In [ ]:
# Inspect a specific reranker log after a failure.
RERANKER_TO_INSPECT = RERANKERS[0]
log_path = OUTPUT_DIR / RERANKER_TO_INSPECT / "run.log"
if log_path.exists():
    print(log_path.read_text()[-8000:])
else:
    print(f"No log found at {log_path}")


In [ ]:
# Optional: rerun a single reranker with a smaller batch size.
# Example:
# RERANKER_BATCH_OVERRIDES['qwen3_reranker_0_6b'] = 4
# single_result = run_one_reranker('qwen3_reranker_0_6b')
# single_result


In [ ]:
# Optional: zip the aggregate directory for download from Colab.
# !zip -r "{OUTPUT_DIR}.zip" "{OUTPUT_DIR}
